# Access the Darkmine Vault API

This notebook demonstrates how to connect to the paid public drilling data API from Darkmine - The Darkmine Vault API. See https://darkmine.ai for details.

Using this free library and the `baselode.adaptors.raw_gswa` helpers, you can access the API, return
pandas DataFrames, and convert those into the canonical baselode data model so they
feed straight into desurveying, strip logs and the rest of `baselode` free library of utility.

Requires:
- a paid Darkmine Vault key (`VAULT_API_KEY` env var)
- `baselode[api]` installed (`pip install -e python/[api]` from this repo, or
  `pip install baselode[api]` from PyPI)

In [ ]:
import json
import os

import pandas as pd

import baselode.adaptors.raw_gswa.api
import baselode.adaptors.raw_gswa.convert
import baselode.adaptors.raw_gswa.queries
import baselode.drill.data
import baselode.extent
import baselode.drill.desurvey

## 1. Construct the API client

`RawGswaApiClient` accepts a base URL and a bearer token; it appends the standard
`/v1/raw/gswa` prefix automatically. The optional `requests` dependency comes in via
`pip install baselode[api]`.

In [ ]:
BASE_URL = os.environ.get('VAULT_API_BASE_URL')
API_KEY = os.environ.get('VAULT_API_KEY')

client = baselode.adaptors.raw_gswa.api.RawGswaApiClient(
    BASE_URL,
    auth_token=API_KEY,
    timeout=180.0,
)

print(json.dumps({
    'base_url': BASE_URL,
    'auth_mode': 'bearer_api_key' if API_KEY else 'unauthenticated',
}, indent=2))

## 2. Discover available tables

`list_tables()` returns the array under `GET /v1/raw/gswa/tables`. `get_schema()`
fetches the per-table column manifest plus the supported query selectors
(`query_methods`).

In [ ]:
tables = client.list_tables()
table_map = {t['name']: t for t in tables}

collar_schema = client.get_schema('dbo_collar')

MAX_DISPLAY = 10
print(json.dumps({
    'table_count': len(tables),
    'first_tables': [t['name'] for t in tables[:MAX_DISPLAY]],
    'dbo_collar_query_methods': collar_schema['query_methods'],
    'dbo_collar_column_count': len(collar_schema['columns']),
}, indent=2))
print('...')

## 3. Bounding-box collar lookup

Build a `baselode.extent.Extent` (axis-aligned bbox + CRS) and pass it as
`extent=` to `fetch_table_rows`. The client validates the CRS and reprojects
the corners to EPSG:4326 if necessary (so it's safe to hand it an Extent in
any projected CRS — MGA, UTM, etc.).

The result is a `pandas` DataFrame whose columns match the API's `columns`
array (raw GSWA shape, PascalCase field names).

In [ ]:
# Use the same example bbox as the bundled demo data — an area in WA around Hyden.
# `Extent` carries its CRS so the API client can validate / reproject it.
tl = [-32.329994174232176, 118.77253985070098]   # top-left  (lat, lon)
br = [-32.75139434476367,  119.74208332736906]   # bottom-right (lat, lon)

EXTENT = baselode.extent.Extent(
    xmin=tl[1], xmax=br[1],   # lon
    ymin=br[0], ymax=tl[0],   # lat (southern hemisphere — both negative)
    name='hyden_bbox', crs="EPSG:4326",
)

raw_collars = client.fetch_table_rows('dbo_collar', extent=EXTENT, limit=10)
raw_collars.head()

Pass `output='geojson'` to get a GeoJSON `FeatureCollection` instead — handy for QGIS
or any GeoJSON-aware tool.

In [ ]:
geojson = client.fetch_table_rows(
    'dbo_collar', extent=EXTENT, limit=2, output='geojson',
)
print(json.dumps(geojson, indent=2)[:1200])

## 4. Pull a hole and convert to baselode shape

Pick the first hole from the bbox query and use the `fetch_*` helpers — they hit the
`/collar-family` endpoint behind the scenes and stitch the right child tables
together so the converters get the long-form rows they expect.

Each `convert_*` returns a baselode-shaped DataFrame with the canonical columns at
the top level and source-only fields folded into the per-row `extra` dict (the
default `extras='bundle'` mode).

In [ ]:
hole_id = raw_collars.iloc[0]['HoleId']
print(f'Working with HoleId: {hole_id}')

raw_collar  = baselode.adaptors.raw_gswa.api.fetch_collars(client, hole_ids=[hole_id])
raw_survey  = baselode.adaptors.raw_gswa.api.fetch_surveys(client, hole_ids=[hole_id])
raw_assays  = baselode.adaptors.raw_gswa.api.fetch_assays_flat(client, hole_ids=[hole_id])
raw_geology = baselode.adaptors.raw_gswa.api.fetch_geology(
    client, hole_ids=[hole_id],
    attribute_columns=['Lith1', 'GeologyComment', 'Weath', 'Regol'], # These are just random attributes for example 
)

collars = baselode.adaptors.raw_gswa.convert.convert_collars(raw_collar)
surveys = baselode.adaptors.raw_gswa.convert.convert_surveys(raw_survey)
assays  = baselode.adaptors.raw_gswa.convert.convert_assays_flat(raw_assays)
geology = baselode.adaptors.raw_gswa.convert.convert_geology(raw_geology)

print(json.dumps({
    'collar_columns':  list(collars.columns),
    'survey_columns':  list(surveys.columns),
    'assay_columns':   list(assays.columns),
    'geology_columns': list(geology.columns),
    'collar_rows':     len(collars),
    'survey_rows':     len(surveys),
    'assay_rows':      len(assays),
    'geology_rows':    len(geology),
}, indent=2))

In [ ]:
collars.head()

Inspect the `extra` dict — it's where GSWA-specific fields like `max_depth`,
`hole_type`, `anumber`, lab metadata, and (for assays) every analyte value end up
by default.

In [ ]:
print('Collar extras:')
print(json.dumps(collars.iloc[0]['extra'], indent=2, default=str))

if not assays.empty:
    print('\nAssay extras (first row, top 10 keys):')
    sample_assay = assays.iloc[0]['extra']
    print(json.dumps(dict(list(sample_assay.items())[:10]), indent=2, default=str))

Need analytes (or any other source-specific column) at the top level for
aggregations? Pass `extras='spread'` and the converter leaves them alongside the
canonical columns instead.

In [ ]:
assays_wide = baselode.adaptors.raw_gswa.convert.convert_assays_flat(
    raw_assays, extras='spread',
)
analyte_cols = [c for c in assays_wide.columns if c.endswith('_PPM')][:8]
assays_wide[['hole_id', 'from', 'to', 'mid', *analyte_cols]].head()

## 5. The `/collar-family` endpoint directly

If you'd rather pull every related table for a hole in a single round-trip and
decide later what to keep, call `fetch_collar_family` directly. Each entry in the
returned `tables` dict is a DataFrame keyed by raw GSWA table name.

In [ ]:
family = client.fetch_collar_family(hole_id=hole_id)

summary = {
    name: {'rows': len(df), 'columns': list(df.columns)[:6]}
    for name, df in family['tables'].items()
    if not df.empty
}
print(json.dumps({
    'matched_collars': family['matched_collar_count'],
    'tables_present':  list(summary.keys()),
}, indent=2))

## 6. Surface samples

The same pattern works for surface geochemistry. Pass an `Extent` to
`fetch_table_rows('gsd_ssassayflat', extent=...)` for an already-pivoted
table, or use `fetch_surface_sample_family(...)` for the long-form EAV path.

In [ ]:
samples_raw = client.fetch_table_rows('gsd_ssassayflat', extent=EXTENT, limit=20)

if samples_raw.empty:
    print('No surface samples found in any candidate extent.')
else:
    samples = baselode.adaptors.raw_gswa.convert.convert_surface_samples_flat(samples_raw)
    print(f'extent: {EXTENT.name}')
    print(f'sample rows: {len(samples)}')
    display(samples.head())

## 7. Downstream: desurvey the hole

The DataFrames are now in the canonical baselode shape — `baselode.drill.desurvey`
consumes them directly. (Skipped automatically if the hole has no survey data.)

In [ ]:
if surveys.empty:
    print('No survey rows for this hole — skipping desurvey.')
else:
    traces = baselode.drill.desurvey.minimum_curvature_desurvey(
        collars, surveys, step=1.0,
    )
    print(f'trace rows: {len(traces)}')
    display(traces.head())